# Surface–Columnar Aerosol Linkage Analysis: Addis Ababa

Implements methods from three key papers to investigate the relationship between
surface aerosol measurements (aethalometer BC, SPARTAN filter chemistry) and
columnar optical properties (AERONET AOD, SSA, FMF, AE).

## Paper Methods Applied

**Eom et al. (2025)** — PM₂.₅/AOD ratio filtering, *w* parameter, dSSA metric

**Segura et al. (2017)** — AOD binning (Δ0.05), boundary layer height stratification

**Snider et al. (2015)** — η decomposition (PM₂.₅/AOD = T1 × T2 × T3)

## Sections
1. BC/AOD Ratio Analysis & Sequential Exclusion (Eom)
2. *w* Parameter: Non-absorbing Fraction (Eom)
3. Fine Soil & dSSA: BC-dominated vs Dust-dominated (Eom)
4. AOD Binning Analysis (Segura)
5. FMF Seasonal Deep Dive
6. η Decomposition: PM₂.₅/AOD (Snider)
7. HIPS vs FTIR Stratified by AOD & BC/AOD
8. SSA Analysis: *w* vs SSA and dSSA vs BC−FS (Eom)
9. Future Work: BLH Stratification (ERA5)

---

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

# --- Project paths ---
FTIR_HIPS_DIR = Path('/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem')
scripts_path = str(FTIR_HIPS_DIR / 'scripts')
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE, FILTER_DATA_PATH
from data_matching import (
    load_aethalometer_data, load_filter_data, match_all_parameters,
    pivot_filter_by_id, load_etad_factors_with_filter_ids
)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 8), 'font.size': 11,
                      'figure.dpi': 100, 'savefig.dpi': 150})

# Output dirs
OUT_PLOTS = 'output/plots/surface_columnar'
OUT_DATA  = 'output/data/surface_columnar'
os.makedirs(OUT_PLOTS, exist_ok=True)
os.makedirs(OUT_DATA, exist_ok=True)

# --- Season config ---
SEASON_MAP = {10: 'Dry', 11: 'Dry', 12: 'Dry', 1: 'Dry', 2: 'Dry',
              3: 'Belg', 4: 'Belg', 5: 'Belg',
              6: 'Kiremt', 7: 'Kiremt', 8: 'Kiremt', 9: 'Kiremt'}
SEASONS_ORDER = ['Dry', 'Belg', 'Kiremt']
SEASON_COLORS = {'Dry': '#E67E22', 'Belg': '#27AE60', 'Kiremt': '#3498DB'}
AERONET_MISSING = -999.

print('Setup complete')

## Data Loading

Load four data sources and merge them on date.

In [ ]:
# ── 1. Aethalometer BC (daily 9am-resampled) ──
BC_PATH = FTIR_HIPS_DIR / 'processed_sites' / 'df_Addis_Ababa_9am_resampled.pkl'
bc_raw = pd.read_pickle(BC_PATH)
bc_raw['datetime_local'] = pd.to_datetime(bc_raw['datetime_local'])
bc_raw.set_index('datetime_local', inplace=True)
bc_raw.sort_index(inplace=True)

# Convert ng/m³ → µg/m³
bc_cols_all = [c for c in bc_raw.columns if 'BCc' in c and 'smoothed' not in c]
for c in bc_cols_all:
    bc_raw[c] = bc_raw[c] / 1000

# QC: remove negatives and >3σ outliers for IR BCc
bc_raw.loc[bc_raw['IR BCc'] < 0, 'IR BCc'] = np.nan
mu, sig = bc_raw['IR BCc'].mean(), bc_raw['IR BCc'].std()
bc_raw.loc[bc_raw['IR BCc'] > mu + 3*sig, 'IR BCc'] = np.nan

bc_raw['Month'] = bc_raw.index.month
bc_raw['Season'] = bc_raw['Month'].map(SEASON_MAP)

# Prepare daily BC for merging (normalize to midnight, strip tz)
bc_daily = bc_raw[['IR BCc', 'UV BCc', 'Season']].copy()
bc_daily.index = bc_daily.index.normalize()
if bc_daily.index.tz is not None:
    bc_daily.index = bc_daily.index.tz_localize(None)
bc_daily.index.name = 'Date'

print(f'BC daily: {bc_daily["IR BCc"].notna().sum()} valid days, '
      f'{bc_daily.index.min().date()} to {bc_daily.index.max().date()}')

In [ ]:
# ── 2. AERONET AOD (Level 2.0 daily) + SDA + SSA Inversion (Level 1.5 daily) ──
AERONET_DIR = ('/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/'
    'My Drive/University/Research/Grad/UC Davis Ann/NASA MAIA/Data/AERONET/Jacros/'
    '20220101_20251231_AAU_Jackros_ET Daily/')

AERONET_AOD_PATH = AERONET_DIR + '20220101_20251231_AAU_Jackros_ET.lev20'
AERONET_SDA_PATH = AERONET_DIR + '20220101_20251231_AAU_Jackros_ET.ONEILL_lev20'
AERONET_SSA_PATH = AERONET_DIR + '20220101_20251231_AAU_Jackros_ET.SSA_ALM_lev15_daily'

aod_df = pd.read_csv(AERONET_AOD_PATH, skiprows=6)
aod_df['Date'] = pd.to_datetime(aod_df['Date(dd:mm:yyyy)'], format='%d:%m:%Y')
aod_df.set_index('Date', inplace=True)
aod_df.sort_index(inplace=True)
aod_df.replace(AERONET_MISSING, np.nan, inplace=True)
aod_df['Season'] = aod_df.index.month.map(SEASON_MAP)

sda_df = pd.read_csv(AERONET_SDA_PATH, skiprows=6)
sda_df['Date'] = pd.to_datetime(sda_df['Date_(dd:mm:yyyy)'], format='%d:%m:%Y')
sda_df.set_index('Date', inplace=True)
sda_df.sort_index(inplace=True)
sda_df.replace(AERONET_MISSING, np.nan, inplace=True)
sda_df['Season'] = sda_df.index.month.map(SEASON_MAP)

# SSA from Level 1.5 Inversion (Almucantar daily averages)
# Note: Level 2.0 SSA requires AOD440 >= 0.4, which Addis rarely meets → all -999.
# Level 1.5 is cloud-cleared and quality-controlled but without the AOD threshold.
ssa_df = pd.read_csv(AERONET_SSA_PATH, skiprows=6)
ssa_df['Date'] = pd.to_datetime(ssa_df['Date(dd:mm:yyyy)'], format='%d:%m:%Y')
ssa_df.set_index('Date', inplace=True)
ssa_df.sort_index(inplace=True)
ssa_df.replace(AERONET_MISSING, np.nan, inplace=True)
ssa_df['Season'] = ssa_df.index.month.map(SEASON_MAP)

# Rename SSA columns for clarity
ssa_rename = {
    'Single_Scattering_Albedo[440nm]': 'SSA_440',
    'Single_Scattering_Albedo[675nm]': 'SSA_675',
    'Single_Scattering_Albedo[870nm]': 'SSA_870',
    'Single_Scattering_Albedo[1020nm]': 'SSA_1020',
}
ssa_df.rename(columns=ssa_rename, inplace=True)

# Compute dSSA = SSA_440 - SSA_870 (Eom et al. metric)
ssa_df['dSSA'] = ssa_df['SSA_440'] - ssa_df['SSA_870']

print(f'AERONET AOD: {len(aod_df)} days  |  SDA: {len(sda_df)} days  |  SSA: {len(ssa_df)} days')
print(f'AOD range: {aod_df.index.min().date()} – {aod_df.index.max().date()}')
print(f'SSA range: {ssa_df.index.min().date()} – {ssa_df.index.max().date()}')
print(f'SSA valid at 440nm: {ssa_df["SSA_440"].notna().sum()} days')

In [ ]:
# ── 3. SPARTAN Filter Chemistry (unified dataset) ──
filter_data = load_filter_data()
etad = filter_data[filter_data['Site'] == 'ETAD'].copy()

# Pivot: one row per filter, columns = parameters
chem_params = [
    'EC_ftir', 'OC_ftir', 'HIPS_Fabs',
    'ChemSpec_Sulfate_Ion_PM2.5', 'ChemSpec_Nitrate_Ion_PM2.5',
    'ChemSpec_Ammonium_Ion_PM2.5',
    'ChemSpec_Iron_PM2.5', 'ChemSpec_Silicon_PM2.5',
    'ChemSpec_Aluminum_PM2.5', 'ChemSpec_Calcium_PM2.5',
    'ChemSpec_Titanium_PM2.5',
    'ChemSpec_BC_PM2.5',
]

chem_pivot = pivot_filter_by_id(filter_data, 'ETAD', params=chem_params)
chem_pivot['date'] = pd.to_datetime(chem_pivot['date'])

# Rename for readability
rename = {
    'chemspec_sulfate_ion_pm2.5': 'sulfate',
    'chemspec_nitrate_ion_pm2.5': 'nitrate',
    'chemspec_ammonium_ion_pm2.5': 'ammonium',
    'chemspec_iron_pm2.5': 'iron',
    'chemspec_silicon_pm2.5': 'silicon',
    'chemspec_aluminum_pm2.5': 'aluminum',
    'chemspec_calcium_pm2.5': 'calcium',
    'chemspec_titanium_pm2.5': 'titanium',
    'chemspec_bc_pm2.5': 'chemspec_bc',
}
chem_pivot.rename(columns=rename, inplace=True)

# Compute derived quantities
# Ammonium sulfate: (NH4)2SO4, molar ratio = 132.14/96.06 ≈ 1.375
chem_pivot['AS'] = chem_pivot['sulfate'] * 1.375
# Ammonium nitrate: NH4NO3, molar ratio = 80.04/62.00 ≈ 1.290
chem_pivot['AN'] = chem_pivot['nitrate'] * 1.290
# Fine soil: 2.2*Al + 2.49*Si + 1.63*Ca + 2.42*Fe + 1.94*Ti (Malm et al. 1994)
chem_pivot['fine_soil'] = (
    2.20 * chem_pivot['aluminum'].fillna(0) +
    2.49 * chem_pivot['silicon'].fillna(0) +
    1.63 * chem_pivot['calcium'].fillna(0) +
    2.42 * chem_pivot['iron'].fillna(0) +
    1.94 * chem_pivot['titanium'].fillna(0)
)
# HIPS BC equivalent
chem_pivot['hips_bc'] = chem_pivot['hips_fabs'] / MAC_VALUE if 'hips_fabs' in chem_pivot.columns else np.nan

print(f'Filter chemistry: {len(chem_pivot)} filters with pivoted data')
print(f'Columns: {list(chem_pivot.columns)}')

In [ ]:
# ── 4. Gravimetric PM2.5 mass (for η decomposition) ──
pm25_data = etad[etad['Parameter'].str.contains('PM2.5', na=False) & 
                  etad['Parameter'].str.contains('Filter', na=False)].copy()
if len(pm25_data) == 0:
    # Try alternative naming
    pm25_data = etad[etad['Parameter'].str.contains('Gravimetric|PM2.5_mass|Total_Mass', na=False, case=False)].copy()

# Check what PM2.5 mass parameters are available
print('Available PM2.5-related parameters for ETAD:')
pm_params = etad[etad['Parameter'].str.contains('PM|mass|grav', case=False, na=False)]['Parameter'].unique()
for p in sorted(pm_params):
    n = len(etad[etad['Parameter'] == p])
    print(f'  {p}: {n} records')

In [ ]:
# ── 5. Build master merged dataset ──
# Merge BC daily + AERONET AOD + AERONET SDA + SSA on date index
master = bc_daily[['IR BCc', 'UV BCc', 'Season']].copy()

# AOD columns
aod_cols = ['AOD_440nm', 'AOD_500nm', 'AOD_675nm', 'AOD_870nm',
            '440-870_Angstrom_Exponent']
master = master.join(aod_df[aod_cols], how='left')

# SDA columns
sda_cols = ['FineModeFraction_500nm[eta]', 'Fine_Mode_AOD_500nm[tau_f]',
            'Coarse_Mode_AOD_500nm[tau_c]']
master = master.join(sda_df[sda_cols], how='left')
master.rename(columns={
    'FineModeFraction_500nm[eta]': 'FMF',
    'Fine_Mode_AOD_500nm[tau_f]': 'Fine_AOD',
    'Coarse_Mode_AOD_500nm[tau_c]': 'Coarse_AOD',
    '440-870_Angstrom_Exponent': 'AE_440_870',
}, inplace=True)

# SSA columns (Level 1.5 inversion)
ssa_cols_merge = ['SSA_440', 'SSA_675', 'SSA_870', 'SSA_1020', 'dSSA']
master = master.join(ssa_df[ssa_cols_merge], how='left')

# Merge filter chemistry by nearest date (±1 day)
chem_indexed = chem_pivot.set_index('date')
chem_indexed.index = pd.to_datetime(chem_indexed.index)
chem_cols_to_merge = ['ftir_ec', 'hips_fabs', 'hips_bc', 'sulfate', 'nitrate',
                       'ammonium', 'iron', 'silicon', 'aluminum', 'fine_soil',
                       'AS', 'AN', 'chemspec_bc']
available_chem = [c for c in chem_cols_to_merge if c in chem_indexed.columns]
master = master.join(chem_indexed[available_chem], how='left')

# Compute key derived columns
master['BC_AOD'] = master['IR BCc'] / master['AOD_500nm']  # BC/AOD ratio
master['dAOD'] = master['AOD_440nm'] - master['AOD_870nm']  # spectral AOD difference

n_bc_aod = master[['IR BCc', 'AOD_500nm']].dropna().shape[0]
n_ssa = master['SSA_440'].notna().sum()
n_chem = master[available_chem].dropna(how='all').shape[0]
print(f'Master dataset: {len(master)} total days')
print(f'  Days with BC + AOD: {n_bc_aod}')
print(f'  Days with SSA: {n_ssa}')
print(f'  Days with any filter chemistry: {n_chem}')

---
## Section 1: BC/AOD Ratio Analysis & Sequential Exclusion

**Method (Eom et al. 2025):** Compute BC/AOD₅₀₀ as a proxy for how well surface
BC represents the aerosol column. Sequentially exclude the bottom 20/40/60/80%
of BC/AOD values and check whether surface–columnar correlations improve.

Days with low BC/AOD may have significant aerosol aloft unrelated to the surface,
muddying comparisons.

In [ ]:
# BC/AOD ratio distribution and seasonal patterns
df1 = master.dropna(subset=['IR BCc', 'AOD_500nm']).copy()
df1 = df1[df1['AOD_500nm'] > 0]  # avoid division by zero

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) BC/AOD histogram by season
ax = axes[0]
for season in SEASONS_ORDER:
    s = df1[df1['Season'] == season]['BC_AOD'].dropna()
    ax.hist(s, bins=25, alpha=0.5, color=SEASON_COLORS[season], label=f'{season} (n={len(s)})')
ax.set_xlabel('BC / AOD₅₀₀ (µg m⁻³ per unitless)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('(a) BC/AOD Ratio Distribution', fontweight='bold')
ax.legend(fontsize=9)

# (b) BC/AOD time series
ax = axes[1]
for season in SEASONS_ORDER:
    s = df1[df1['Season'] == season]
    ax.scatter(s.index, s['BC_AOD'], s=15, alpha=0.6,
              color=SEASON_COLORS[season], label=season)
ax.set_ylabel('BC / AOD₅₀₀', fontsize=11)
ax.set_title('(b) BC/AOD Time Series', fontweight='bold')
ax.legend(fontsize=9)

# (c) Seasonal boxplot
ax = axes[2]
data_box = [df1[df1['Season'] == s]['BC_AOD'].dropna() for s in SEASONS_ORDER]
bp = ax.boxplot(data_box, labels=SEASONS_ORDER, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=3, alpha=0.3))
for patch, s in zip(bp['boxes'], SEASONS_ORDER):
    patch.set_facecolor(SEASON_COLORS[s])
    patch.set_alpha(0.7)
ax.set_ylabel('BC / AOD₅₀₀', fontsize=11)
ax.set_title('(c) BC/AOD by Season', fontweight='bold')

plt.suptitle('BC/AOD Ratio — Surface Representativeness Proxy (Eom et al.)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/bc_aod_ratio_overview.png', bbox_inches='tight')
plt.show()

# Summary stats
print('\nBC/AOD ratio statistics:')
for season in SEASONS_ORDER:
    s = df1[df1['Season'] == season]['BC_AOD']
    print(f'  {season}: n={len(s)}, median={s.median():.2f}, mean={s.mean():.2f}, std={s.std():.2f}')

In [ ]:
# Sequential exclusion: remove bottom X% of BC/AOD and recompute correlations
df_ex = master.dropna(subset=['IR BCc', 'AOD_500nm']).copy()
df_ex = df_ex[df_ex['AOD_500nm'] > 0]

exclusion_levels = [0, 20, 40, 60, 80]
results = []

for ex in exclusion_levels:
    if ex == 0:
        subset = df_ex.copy()
    else:
        threshold = df_ex['BC_AOD'].quantile(ex / 100)
        subset = df_ex[df_ex['BC_AOD'] >= threshold].copy()
    
    n = len(subset)
    if n < 5:
        continue
    
    # Correlations: BC vs AOD, BC vs Fine_AOD, BC vs AE
    row = {'exclusion': f'Ex{ex}', 'n': n}
    for yvar, ylabel in [('AOD_500nm', 'AOD500'), ('AE_440_870', 'AE'),
                          ('FMF', 'FMF'), ('Fine_AOD', 'Fine AOD')]:
        valid = subset.dropna(subset=['IR BCc', yvar])
        if len(valid) >= 5:
            r, p = stats.pearsonr(valid['IR BCc'], valid[yvar])
            row[f'r_{ylabel}'] = r
            row[f'p_{ylabel}'] = p
            row[f'n_{ylabel}'] = len(valid)
    results.append(row)

results_df = pd.DataFrame(results)
print('Sequential Exclusion Results (Eom et al. approach):')
print('=' * 80)
display_cols = ['exclusion', 'n'] + [c for c in results_df.columns if c.startswith('r_')]
print(results_df[display_cols].to_string(index=False, float_format='%.3f'))

# Plot correlation improvement
fig, ax = plt.subplots(figsize=(10, 6))
r_cols = [c for c in results_df.columns if c.startswith('r_')]
markers = ['o', 's', '^', 'D']
for i, rc in enumerate(r_cols):
    label = rc.replace('r_', '')
    ax.plot(exclusion_levels[:len(results_df)], results_df[rc], f'-{markers[i]}',
            markersize=8, linewidth=2, label=label)

ax.set_xlabel('Bottom % of BC/AOD Excluded', fontsize=12)
ax.set_ylabel('Pearson r (BC vs columnar property)', fontsize=12)
ax.set_title('Sequential Exclusion: Correlation Improvement\n(Eom et al. 2025 method)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xticks(exclusion_levels)
ax.set_xticklabels([f'Ex{e}' for e in exclusion_levels])
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/sequential_exclusion.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter plots at Ex0 vs Ex60 for visual comparison
df_ex0 = master.dropna(subset=['IR BCc', 'AOD_500nm']).copy()
df_ex0 = df_ex0[df_ex0['AOD_500nm'] > 0]

thresh60 = df_ex0['BC_AOD'].quantile(0.60)
df_ex60 = df_ex0[df_ex0['BC_AOD'] >= thresh60].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for col_idx, (df_sub, label) in enumerate([(df_ex0, 'Ex0 (all data)'), (df_ex60, 'Ex60')]):
    for row_idx, (ycol, ylabel) in enumerate([('AOD_500nm', 'AOD 500nm'), ('AE_440_870', 'AE 440-870')]):
        ax = axes[row_idx, col_idx]
        valid = df_sub.dropna(subset=['IR BCc', ycol])
        for season in SEASONS_ORDER:
            s = valid[valid['Season'] == season]
            ax.scatter(s['IR BCc'], s[ycol], s=25, alpha=0.5,
                      color=SEASON_COLORS[season], edgecolors='k', linewidth=0.2, label=season)
        if len(valid) >= 5:
            r, p = stats.pearsonr(valid['IR BCc'], valid[ycol])
            # Regression line
            z = np.polyfit(valid['IR BCc'], valid[ycol], 1)
            xr = np.linspace(valid['IR BCc'].min(), valid['IR BCc'].max(), 50)
            ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.7)
            ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
                   transform=ax.transAxes, va='top', fontsize=9,
                   bbox=dict(boxstyle='round', fc='white', alpha=0.9))
        ax.set_xlabel('BC (µg/m³)', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(f'{ylabel} vs BC — {label}', fontweight='bold', fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

plt.suptitle('Effect of BC/AOD Exclusion on Surface–Column Correlations',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/exclusion_scatter_comparison.png', bbox_inches='tight')
plt.show()

---
## Section 2: *w* Parameter — Non-absorbing Fraction (Eom et al.)

**w = (AS + AN) / (AS + AN + BC)**

where AS = ammonium sulfate, AN = ammonium nitrate, BC = black carbon.

This ratio describes the fraction of fine-mode mass that is non-absorbing.
Higher *w* → scattering-dominated aerosol. Lower *w* → absorption-dominated.

Eom et al. plotted *w* against AERONET SSA — we lack SSA (inversion product
not downloaded), so instead we examine *w* seasonally and against AE/AOD.

In [ ]:
# Compute w parameter from filter chemistry
df_w = chem_pivot.dropna(subset=['AS', 'AN']).copy()

# Use chemspec_bc if available, else FTIR EC, else HIPS BC
bc_col_for_w = None
for candidate in ['chemspec_bc', 'ftir_ec', 'hips_bc']:
    if candidate in df_w.columns and df_w[candidate].notna().sum() > 5:
        bc_col_for_w = candidate
        break
print(f'Using {bc_col_for_w} as BC for w parameter')

df_w = df_w.dropna(subset=[bc_col_for_w])
denom = df_w['AS'] + df_w['AN'] + df_w[bc_col_for_w]
df_w['w'] = (df_w['AS'] + df_w['AN']) / denom
df_w = df_w[denom > 0]  # avoid division by zero
df_w['Month'] = df_w['date'].dt.month
df_w['Season'] = df_w['Month'].map(SEASON_MAP)

print(f'\nw parameter computed for {len(df_w)} filters')
print(f'  w range: {df_w["w"].min():.3f} – {df_w["w"].max():.3f}')
print(f'  w mean:  {df_w["w"].mean():.3f} ± {df_w["w"].std():.3f}')

# Merge w with AERONET
df_w_aeronet = df_w.set_index('date')[['w', bc_col_for_w, 'AS', 'AN', 'Season']].copy()
df_w_aeronet = df_w_aeronet.join(aod_df[['AOD_500nm', '440-870_Angstrom_Exponent']], how='inner')
df_w_aeronet.rename(columns={'440-870_Angstrom_Exponent': 'AE'}, inplace=True)
print(f'w + AERONET matched: {len(df_w_aeronet)} days')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (a) w by season
ax = axes[0, 0]
box_data = [df_w[df_w['Season'] == s]['w'].dropna() for s in SEASONS_ORDER]
bp = ax.boxplot(box_data, labels=SEASONS_ORDER, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=4, alpha=0.4))
for patch, s in zip(bp['boxes'], SEASONS_ORDER):
    patch.set_facecolor(SEASON_COLORS[s])
    patch.set_alpha(0.7)
for i, s in enumerate(SEASONS_ORDER):
    n = len(df_w[df_w['Season'] == s]['w'].dropna())
    ax.text(i+1, ax.get_ylim()[0] + 0.02, f'n={n}', ha='center', fontsize=9)
ax.set_ylabel('w = (AS+AN) / (AS+AN+BC)', fontsize=11)
ax.set_title('(a) Non-absorbing Fraction by Season', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# (b) w vs AOD
ax = axes[0, 1]
valid = df_w_aeronet.dropna(subset=['w', 'AOD_500nm'])
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['w'], s['AOD_500nm'], s=40, alpha=0.6,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['w'], valid['AOD_500nm'])
    ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
           transform=ax.transAxes, va='top', fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('w', fontsize=11)
ax.set_ylabel('AOD 500nm', fontsize=11)
ax.set_title('(b) w vs AOD₅₀₀', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (c) w vs AE
ax = axes[1, 0]
valid = df_w_aeronet.dropna(subset=['w', 'AE'])
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['w'], s['AE'], s=40, alpha=0.6,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['w'], valid['AE'])
    ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
           transform=ax.transAxes, va='top', fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('w', fontsize=11)
ax.set_ylabel('Ångström Exponent (440-870)', fontsize=11)
ax.set_title('(c) w vs AE', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (d) Component breakdown: AS, AN, BC stacked bars by season
ax = axes[1, 1]
season_means = []
for s in SEASONS_ORDER:
    sub = df_w[df_w['Season'] == s]
    season_means.append({
        'Season': s,
        'AS': sub['AS'].mean(),
        'AN': sub['AN'].mean(),
        'BC': sub[bc_col_for_w].mean()
    })
sm = pd.DataFrame(season_means)
x = np.arange(len(SEASONS_ORDER))
width = 0.5
ax.bar(x, sm['AS'], width, label='Amm. Sulfate', color='#3498DB', alpha=0.8)
ax.bar(x, sm['AN'], width, bottom=sm['AS'], label='Amm. Nitrate', color='#2ECC71', alpha=0.8)
ax.bar(x, sm['BC'], width, bottom=sm['AS']+sm['AN'], label='BC', color='#2C3E50', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(SEASONS_ORDER)
ax.set_ylabel('Concentration (µg/m³)', fontsize=11)
ax.set_title('(d) AS + AN + BC Composition by Season', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('w Parameter: Non-absorbing Mass Fraction (Eom et al.)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/w_parameter_analysis.png', bbox_inches='tight')
plt.show()

# Stats by season
print('\nw parameter by season:')
for s in SEASONS_ORDER:
    sub = df_w[df_w['Season'] == s]
    if len(sub) > 0:
        print(f'  {s}: n={len(sub)}, w={sub["w"].mean():.3f}±{sub["w"].std():.3f}, '
              f'AS={sub["AS"].mean():.2f}, AN={sub["AN"].mean():.2f}, BC={sub[bc_col_for_w].mean():.2f}')

---
## Section 3: Fine Soil & Absorption Proxy — dAOD vs BC−FS

**Method (Eom et al.):** dSSA = SSA₄₄₀ − SSA₈₇₀ plotted against BC − Fine Soil mass.

Since we lack SSA (AERONET inversion data not yet downloaded), we use
**dAOD = AOD₄₄₀ − AOD₈₇₀** as a proxy for wavelength-dependent extinction,
which correlates with absorption characteristics.

- BC-dominated: dAOD should be positive (stronger absorption at shorter λ)
- Dust-dominated: dAOD flatter or inverted

Fine soil computed as: 2.2×Al + 2.49×Si + 1.63×Ca + 2.42×Fe + 1.94×Ti (Malm 1994)

In [ ]:
# Merge filter chemistry with AERONET for the dAOD analysis
chem_aeronet = chem_pivot.set_index('date').copy()
chem_aeronet.index = pd.to_datetime(chem_aeronet.index)
chem_aeronet = chem_aeronet.join(aod_df[['AOD_440nm', 'AOD_870nm']], how='inner')
chem_aeronet['dAOD'] = chem_aeronet['AOD_440nm'] - chem_aeronet['AOD_870nm']
chem_aeronet['Month'] = chem_aeronet.index.month
chem_aeronet['Season'] = chem_aeronet['Month'].map(SEASON_MAP)

# Use FTIR EC as BC proxy for BC - Fine Soil difference
bc_for_diff = 'ftir_ec' if 'ftir_ec' in chem_aeronet.columns else 'hips_bc'
chem_aeronet['BC_minus_FS'] = chem_aeronet[bc_for_diff] - chem_aeronet['fine_soil']

valid = chem_aeronet.dropna(subset=['dAOD', 'BC_minus_FS'])
print(f'dAOD vs BC−FS: {len(valid)} matched filter-days')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# (a) dAOD vs BC - Fine Soil, colored by season
ax = axes[0]
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['BC_minus_FS'], s['dAOD'], s=50, alpha=0.6,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['BC_minus_FS'], valid['dAOD'])
    ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
           transform=ax.transAxes, va='top', fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel(f'{bc_for_diff} − Fine Soil (µg/m³)', fontsize=11)
ax.set_ylabel('dAOD = AOD₄₄₀ − AOD₈₇₀', fontsize=11)
ax.set_title('(a) dAOD vs BC−FS (Eom et al. proxy)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (b) Fine Soil vs BC by season
ax = axes[1]
valid2 = chem_aeronet.dropna(subset=[bc_for_diff, 'fine_soil'])
for season in SEASONS_ORDER:
    s = valid2[valid2['Season'] == season]
    ax.scatter(s['fine_soil'], s[bc_for_diff], s=50, alpha=0.6,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
lim = max(valid2['fine_soil'].max(), valid2[bc_for_diff].max()) * 1.1
ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.4, label='1:1')
ax.set_xlabel('Fine Soil (µg/m³)', fontsize=11)
ax.set_ylabel(f'{bc_for_diff} (µg/m³)', fontsize=11)
ax.set_title('(b) BC vs Fine Soil by Season', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (c) BC / Fine Soil ratio by season (BC-dominated vs Dust-dominated classification)
ax = axes[2]
valid3 = chem_aeronet.dropna(subset=[bc_for_diff, 'fine_soil'])
valid3 = valid3[valid3['fine_soil'] > 0]
valid3['BC_FS_ratio'] = valid3[bc_for_diff] / valid3['fine_soil']
box_data = [valid3[valid3['Season'] == s]['BC_FS_ratio'].dropna() for s in SEASONS_ORDER]
bp = ax.boxplot(box_data, labels=SEASONS_ORDER, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=4, alpha=0.4))
for patch, s in zip(bp['boxes'], SEASONS_ORDER):
    patch.set_facecolor(SEASON_COLORS[s])
    patch.set_alpha(0.7)
ax.axhline(1, color='red', linewidth=1, linestyle='--', alpha=0.6, label='BC = FS')
ax.set_ylabel('BC / Fine Soil Ratio', fontsize=11)
ax.set_title('(c) BC vs Dust Dominance by Season', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Absorption vs Dust Dominance Analysis (Modified Eom et al.)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/daod_bc_fine_soil.png', bbox_inches='tight')
plt.show()

# Is Addis BC-dominated or dust-dominated?
print('\nBC / Fine Soil ratio by season:')
for s in SEASONS_ORDER:
    sub = valid3[valid3['Season'] == s]['BC_FS_ratio']
    if len(sub) > 0:
        pct_bc_dom = (sub > 1).mean() * 100
        print(f'  {s}: n={len(sub)}, median={sub.median():.2f}, '
              f'mean={sub.mean():.2f}, BC-dominated: {pct_bc_dom:.0f}%')

---
## Section 4: AOD Binning Analysis (Segura et al.)

**Method:** Bin AOD in intervals of 0.05 and compute mean BC within each bin.
Segura et al. showed this dramatically improves PM-AOD correlations
(from r~0.3 daily scatter to r>0.95 for binned).

Also check if the HIPS-FTIR discrepancy is concentrated at certain AOD levels.

In [ ]:
# AOD binning (0.05 intervals)
df_bin = master.dropna(subset=['IR BCc', 'AOD_500nm']).copy()
df_bin = df_bin[df_bin['AOD_500nm'] > 0]

bin_width = 0.05
max_aod = df_bin['AOD_500nm'].quantile(0.98)
bins = np.arange(0, max_aod + bin_width, bin_width)
df_bin['AOD_bin'] = pd.cut(df_bin['AOD_500nm'], bins=bins)
df_bin['AOD_bin_mid'] = df_bin['AOD_bin'].apply(lambda x: x.mid if pd.notna(x) else np.nan)

# Binned means
binned = df_bin.groupby('AOD_bin_mid').agg(
    BC_mean=('IR BCc', 'mean'),
    BC_std=('IR BCc', 'std'),
    BC_se=('IR BCc', lambda x: x.std() / np.sqrt(len(x)) if len(x) > 1 else np.nan),
    AOD_mean=('AOD_500nm', 'mean'),
    n=('IR BCc', 'count')
).dropna()

# Only bins with ≥3 points
binned_valid = binned[binned['n'] >= 3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# (a) Daily scatter with binned means overlay
ax = axes[0]
for season in SEASONS_ORDER:
    s = df_bin[df_bin['Season'] == season]
    ax.scatter(s['AOD_500nm'], s['IR BCc'], s=15, alpha=0.3,
              color=SEASON_COLORS[season], label=f'{season} daily')
ax.errorbar(binned_valid.index, binned_valid['BC_mean'], yerr=binned_valid['BC_se'],
           fmt='ko-', markersize=8, linewidth=2, capsize=3, label='Binned mean ± SE',
           zorder=10)
if len(binned_valid) >= 3:
    r_bin, p_bin = stats.pearsonr(binned_valid.index, binned_valid['BC_mean'])
    r_daily, p_daily = stats.pearsonr(df_bin['AOD_500nm'], df_bin['IR BCc'])
    ax.text(0.05, 0.95, f'Daily: r={r_daily:.3f}\nBinned: r={r_bin:.3f}',
           transform=ax.transAxes, va='top', fontsize=10,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('AOD 500nm', fontsize=11)
ax.set_ylabel('BC (µg/m³)', fontsize=11)
ax.set_title('(a) BC vs AOD: Daily + Binned Means', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (b) Number of observations per bin
ax = axes[1]
ax.bar(binned.index, binned['n'], width=bin_width*0.8, color='#34495E', alpha=0.7,
       edgecolor='white')
ax.set_xlabel('AOD 500nm (bin center)', fontsize=11)
ax.set_ylabel('Number of days', fontsize=11)
ax.set_title('(b) Data Density per AOD Bin', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# (c) Binned analysis by season
ax = axes[2]
for season in SEASONS_ORDER:
    s_data = df_bin[df_bin['Season'] == season]
    s_binned = s_data.groupby('AOD_bin_mid').agg(
        BC_mean=('IR BCc', 'mean'), n=('IR BCc', 'count')
    ).dropna()
    s_valid = s_binned[s_binned['n'] >= 2]
    if len(s_valid) >= 2:
        ax.plot(s_valid.index, s_valid['BC_mean'], 'o-', markersize=6, linewidth=1.5,
               color=SEASON_COLORS[season], label=season)
ax.set_xlabel('AOD 500nm (bin center)', fontsize=11)
ax.set_ylabel('Mean BC (µg/m³)', fontsize=11)
ax.set_title('(c) Binned BC–AOD by Season', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('AOD Binning Analysis (Segura et al. method, Δ=0.05)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/aod_binning.png', bbox_inches='tight')
plt.show()

# Print correlation improvement
print(f'\nCorrelation improvement with binning:')
print(f'  Daily scatter:  r = {r_daily:.3f} (n={len(df_bin)})')
print(f'  Binned (Δ0.05): r = {r_bin:.3f} (n={len(binned_valid)} bins)')

---
## Section 5: FMF Seasonal Deep Dive

**Eom et al.** used FMF from AERONET but it wasn't explored deeply in existing notebooks.

If kiremt (biomass burning season) has distinctly different FMF, that indicates
particle size shifts that could affect MAC values and HIPS measurements.

In [ ]:
# FMF seasonal analysis + relationship with BC and source apportionment
df_fmf = master.dropna(subset=['FMF']).copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (a) FMF by month
ax = axes[0, 0]
monthly = df_fmf.groupby(df_fmf.index.month)['FMF'].agg(['mean', 'std', 'count'])
colors_monthly = [SEASON_COLORS[SEASON_MAP[m]] for m in monthly.index]
ax.bar(monthly.index, monthly['mean'], yerr=monthly['std']/np.sqrt(monthly['count']),
       color=colors_monthly, alpha=0.7, edgecolor='black', linewidth=0.5, capsize=3)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_ylabel('Fine Mode Fraction', fontsize=11)
ax.set_title('(a) FMF by Month', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# (b) FMF vs BC scatter by season
ax = axes[0, 1]
valid = df_fmf.dropna(subset=['IR BCc', 'FMF'])
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['IR BCc'], s['FMF'], s=30, alpha=0.5,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.2, label=season)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['IR BCc'], valid['FMF'])
    ax.text(0.05, 0.05, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
           transform=ax.transAxes, fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('BC (µg/m³)', fontsize=11)
ax.set_ylabel('FMF', fontsize=11)
ax.set_title('(b) FMF vs Surface BC', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (c) FMF vs AE (should correlate — both relate to size)
ax = axes[1, 0]
valid = df_fmf.dropna(subset=['FMF', 'AE_440_870'])
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['AE_440_870'], s['FMF'], s=30, alpha=0.5,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.2, label=season)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['AE_440_870'], valid['FMF'])
    ax.text(0.05, 0.05, f'r={r:.3f}, p={p:.2e}',
           transform=ax.transAxes, fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('Ångström Exponent (440-870)', fontsize=11)
ax.set_ylabel('FMF', fontsize=11)
ax.set_title('(c) FMF vs AE (size consistency check)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (d) FMF vs Fine/Coarse AOD ratio
ax = axes[1, 1]
valid = df_fmf.dropna(subset=['FMF', 'Fine_AOD', 'Coarse_AOD'])
valid = valid[valid['Coarse_AOD'] > 0]
valid['FC_ratio'] = valid['Fine_AOD'] / valid['Coarse_AOD']
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['FMF'], s['FC_ratio'], s=30, alpha=0.5,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.2, label=season)
ax.set_xlabel('FMF', fontsize=11)
ax.set_ylabel('Fine AOD / Coarse AOD', fontsize=11)
ax.set_title('(d) FMF vs Fine/Coarse AOD Ratio', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('Fine Mode Fraction Seasonal Deep Dive',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/fmf_seasonal_deep_dive.png', bbox_inches='tight')
plt.show()

# Kruskal-Wallis test for seasonal FMF differences
groups = [df_fmf[df_fmf['Season'] == s]['FMF'].dropna() for s in SEASONS_ORDER]
groups_valid = [g for g in groups if len(g) >= 3]
if len(groups_valid) >= 2:
    H, p_kw = stats.kruskal(*groups_valid)
    print(f'\nKruskal-Wallis test for seasonal FMF differences: H={H:.2f}, p={p_kw:.4e}')

print('\nFMF by season:')
for s in SEASONS_ORDER:
    sub = df_fmf[df_fmf['Season'] == s]['FMF']
    print(f'  {s}: n={len(sub)}, mean={sub.mean():.3f}±{sub.std():.3f}, median={sub.median():.3f}')

---
## Section 6: η Decomposition — PM₂.₅/AOD (Snider et al. 2015)

**η = PM₂.₅ / AOD₅₀₀**

Snider et al. decomposed η into three terms:
- **T1**: Effective scale height of aerosol (boundary layer depth effect)
- **T2**: Diurnal variation in ground-level scatter
- **T3**: Mass scattering/extinction efficiency

We approximate η using aethalometer BC as a PM₂.₅ proxy (BC/AOD) and, where
available, filter gravimetric PM₂.₅ mass. We analyze which factors dominate
the variability of η at Addis Ababa.

In [ ]:
# η analysis using BC as PM2.5 proxy
df_eta = master.dropna(subset=['IR BCc', 'AOD_500nm']).copy()
df_eta = df_eta[df_eta['AOD_500nm'] > 0]
df_eta['eta_bc'] = df_eta['IR BCc'] / df_eta['AOD_500nm']  # µg/m³ per AOD

# Also try with filter PM2.5 if available
# Look for gravimetric mass in filter data
pm25_param = etad[etad['Parameter'].str.contains('PM2.5', na=False) & 
                   ~etad['Parameter'].str.contains('ChemSpec|HIPS|EC_ftir|OC_ftir|OM', na=False)]
has_grav_pm25 = False
if len(pm25_param) > 0:
    pm25_by_date = pm25_param.groupby('SampleDate')['Concentration'].first()
    pm25_by_date.index = pd.to_datetime(pm25_by_date.index)
    df_eta = df_eta.join(pm25_by_date.rename('PM25_grav'), how='left')
    df_eta['eta_pm25'] = df_eta['PM25_grav'] / df_eta['AOD_500nm']
    has_grav_pm25 = df_eta['eta_pm25'].notna().sum() > 5
    print(f'Gravimetric PM2.5 available: {df_eta["eta_pm25"].notna().sum()} days')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# (a) η_BC distribution by season
ax = axes[0]
box_data = [df_eta[df_eta['Season'] == s]['eta_bc'].dropna() for s in SEASONS_ORDER]
bp = ax.boxplot(box_data, labels=SEASONS_ORDER, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=3, alpha=0.3))
for patch, s in zip(bp['boxes'], SEASONS_ORDER):
    patch.set_facecolor(SEASON_COLORS[s])
    patch.set_alpha(0.7)
ax.set_ylabel('η_BC = BC / AOD₅₀₀ (µg m⁻³)', fontsize=11)
ax.set_title('(a) η_BC by Season', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# (b) η_BC time series
ax = axes[1]
for season in SEASONS_ORDER:
    s = df_eta[df_eta['Season'] == season]
    ax.scatter(s.index, s['eta_bc'], s=15, alpha=0.5,
              color=SEASON_COLORS[season], label=season)
rolling_eta = df_eta['eta_bc'].rolling(30, min_periods=7).mean()
ax.plot(rolling_eta.index, rolling_eta.values, 'k-', linewidth=2, alpha=0.7, label='30-day mean')
ax.set_ylabel('η_BC = BC / AOD₅₀₀', fontsize=11)
ax.set_title('(b) η Time Series', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (c) η_BC vs meteorological proxies
ax = axes[2]
# Use precipitable water as proxy for boundary layer moisture/depth
pw_col = 'Precipitable_Water(cm)'
df_eta_pw = df_eta.join(aod_df[[pw_col]], how='left').dropna(subset=['eta_bc', pw_col])
for season in SEASONS_ORDER:
    s = df_eta_pw[df_eta_pw['Season'] == season]
    ax.scatter(s[pw_col], s['eta_bc'], s=30, alpha=0.5,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.2, label=season)
if len(df_eta_pw) >= 5:
    r, p = stats.pearsonr(df_eta_pw[pw_col], df_eta_pw['eta_bc'])
    ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}\nn={len(df_eta_pw)}',
           transform=ax.transAxes, va='top', fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('Precipitable Water (cm)', fontsize=11)
ax.set_ylabel('η_BC = BC / AOD₅₀₀', fontsize=11)
ax.set_title('(c) η vs PW (BLH proxy)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('η = PM₂.₅/AOD Decomposition (Snider et al. approach)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/eta_decomposition.png', bbox_inches='tight')
plt.show()

print('\nη_BC (BC/AOD) statistics by season:')
for s in SEASONS_ORDER:
    sub = df_eta[df_eta['Season'] == s]['eta_bc']
    print(f'  {s}: n={len(sub)}, mean={sub.mean():.2f}±{sub.std():.2f}, median={sub.median():.2f}')

# Coefficient of variation to identify dominant source of η variability
print(f'\nη_BC coefficient of variation: {df_eta["eta_bc"].std()/df_eta["eta_bc"].mean():.2f}')
print(f'BC coefficient of variation:   {df_eta["IR BCc"].std()/df_eta["IR BCc"].mean():.2f}')
print(f'AOD coefficient of variation:  {df_eta["AOD_500nm"].std()/df_eta["AOD_500nm"].mean():.2f}')

---
## Section 7: HIPS vs FTIR Stratified by AOD & BC/AOD

Test whether the HIPS-FTIR discrepancy changes with:
- AOD level (high AOD = more aerosol, potentially more column vs surface decoupling)
- BC/AOD ratio (surface representativeness)
- Season

If the discrepancy varies with these, it suggests a physical (not instrumental) driver.

In [ ]:
# HIPS vs FTIR stratified by AOD and BC/AOD
df_hf = master.dropna(subset=['hips_bc', 'ftir_ec']).copy()
df_hf['hips_ftir_ratio'] = df_hf['hips_bc'] / df_hf['ftir_ec']
df_hf['hips_ftir_diff'] = df_hf['hips_bc'] - df_hf['ftir_ec']

print(f'HIPS vs FTIR comparison: {len(df_hf)} matched filter-days')
print(f'  HIPS/FTIR ratio: mean={df_hf["hips_ftir_ratio"].mean():.2f}, '
      f'median={df_hf["hips_ftir_ratio"].median():.2f}')

# Subset with AERONET data
df_hf_aod = df_hf.dropna(subset=['AOD_500nm'])
print(f'  With AOD data: {len(df_hf_aod)} days')

if len(df_hf_aod) >= 6:
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    
    # (a) HIPS vs FTIR scatter colored by AOD
    ax = axes[0, 0]
    sc = ax.scatter(df_hf_aod['ftir_ec'], df_hf_aod['hips_bc'],
                   c=df_hf_aod['AOD_500nm'], cmap='YlOrRd', s=50, alpha=0.7,
                   edgecolors='k', linewidth=0.3)
    plt.colorbar(sc, ax=ax, label='AOD 500nm')
    lim = max(df_hf_aod['ftir_ec'].max(), df_hf_aod['hips_bc'].max()) * 1.1
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, alpha=0.5)
    ax.set_xlabel('FTIR EC (µg/m³)', fontsize=11)
    ax.set_ylabel('HIPS BC (µg/m³)', fontsize=11)
    ax.set_title('(a) HIPS vs FTIR, colored by AOD', fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    
    # (b) HIPS/FTIR ratio vs AOD
    ax = axes[0, 1]
    for season in SEASONS_ORDER:
        s = df_hf_aod[df_hf_aod['Season'] == season]
        ax.scatter(s['AOD_500nm'], s['hips_ftir_ratio'], s=40, alpha=0.6,
                  color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
    ax.axhline(1, color='red', linewidth=1, linestyle='--', alpha=0.6)
    if len(df_hf_aod) >= 5:
        r, p = stats.pearsonr(df_hf_aod['AOD_500nm'], df_hf_aod['hips_ftir_ratio'])
        ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}',
               transform=ax.transAxes, va='top', fontsize=9,
               bbox=dict(boxstyle='round', fc='white', alpha=0.9))
    ax.set_xlabel('AOD 500nm', fontsize=11)
    ax.set_ylabel('HIPS BC / FTIR EC', fontsize=11)
    ax.set_title('(b) HIPS/FTIR Ratio vs AOD', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # (c) HIPS/FTIR ratio vs BC/AOD
    ax = axes[1, 0]
    valid = df_hf_aod.dropna(subset=['BC_AOD'])
    for season in SEASONS_ORDER:
        s = valid[valid['Season'] == season]
        ax.scatter(s['BC_AOD'], s['hips_ftir_ratio'], s=40, alpha=0.6,
                  color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
    ax.axhline(1, color='red', linewidth=1, linestyle='--', alpha=0.6)
    if len(valid) >= 5:
        r, p = stats.pearsonr(valid['BC_AOD'], valid['hips_ftir_ratio'])
        ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}',
               transform=ax.transAxes, va='top', fontsize=9,
               bbox=dict(boxstyle='round', fc='white', alpha=0.9))
    ax.set_xlabel('BC / AOD₅₀₀', fontsize=11)
    ax.set_ylabel('HIPS BC / FTIR EC', fontsize=11)
    ax.set_title('(c) HIPS/FTIR Ratio vs BC/AOD', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # (d) HIPS/FTIR ratio by season (with AOD bin split if enough data)
    ax = axes[1, 1]
    aod_median = df_hf_aod['AOD_500nm'].median()
    df_hf_aod['AOD_group'] = np.where(df_hf_aod['AOD_500nm'] <= aod_median, 'Low AOD', 'High AOD')
    
    positions = []
    labels = []
    box_data = []
    colors_box = []
    pos = 1
    for season in SEASONS_ORDER:
        for grp, offset, alpha in [('Low AOD', 0, 0.5), ('High AOD', 0.35, 0.8)]:
            sub = df_hf_aod[(df_hf_aod['Season'] == season) & (df_hf_aod['AOD_group'] == grp)]
            if len(sub) >= 2:
                box_data.append(sub['hips_ftir_ratio'].dropna())
                positions.append(pos + offset)
                labels.append(f'{season[:3]}\n{grp[:4]}')
                colors_box.append(SEASON_COLORS[season])
        pos += 1
    
    if box_data:
        bp = ax.boxplot(box_data, positions=positions, widths=0.25, patch_artist=True,
                       showfliers=True, flierprops=dict(marker='.', markersize=3))
        for patch, c in zip(bp['boxes'], colors_box):
            patch.set_facecolor(c)
            patch.set_alpha(0.7)
    ax.axhline(1, color='red', linewidth=1, linestyle='--', alpha=0.6)
    ax.set_ylabel('HIPS BC / FTIR EC', fontsize=11)
    ax.set_title(f'(d) HIPS/FTIR by Season × AOD (split at {aod_median:.2f})', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('HIPS–FTIR Discrepancy: AOD & BC/AOD Stratification',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{OUT_PLOTS}/hips_ftir_aod_stratification.png', bbox_inches='tight')
    plt.show()
else:
    print('Not enough matched HIPS-FTIR + AERONET data for stratified analysis')

# Stats
print('\nHIPS/FTIR ratio by season:')
for s in SEASONS_ORDER:
    sub = df_hf[df_hf['Season'] == s]['hips_ftir_ratio']
    if len(sub) > 0:
        print(f'  {s}: n={len(sub)}, mean={sub.mean():.2f}±{sub.std():.2f}, median={sub.median():.2f}')

---
## Section 8: SSA Analysis — *w* vs SSA and dSSA vs BC−FS (Eom et al.)

Now using actual AERONET Level 1.5 Inversion SSA data (Almucantar daily averages).

**Note:** Level 2.0 SSA requires AOD₄₄₀ ≥ 0.4, which Addis Ababa rarely meets.
Level 1.5 is cloud-cleared and quality-controlled but without this AOD threshold,
providing ~629 daily SSA observations.

### Key analyses:
- **dSSA = SSA₄₄₀ − SSA₈₇₀** vs BC − Fine Soil mass (Eom et al. Fig.)
- **w vs SSA** at multiple wavelengths — does Addis follow the 14-site global pattern?
- **SSA seasonal patterns** — BC-dominated vs dust-dominated optical character

In [ ]:
# ── 8a. SSA Overview and Seasonal Patterns ──
df_ssa = master.dropna(subset=['SSA_440']).copy()
print(f'SSA data: {len(df_ssa)} days with valid SSA at 440nm')

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# (a) SSA at all wavelengths - seasonal boxplots
ax = axes[0, 0]
wls = [('SSA_440', '440nm', '#9C27B0'), ('SSA_675', '675nm', '#4CAF50'),
       ('SSA_870', '870nm', '#F44336'), ('SSA_1020', '1020nm', '#795548')]
positions = np.arange(len(SEASONS_ORDER))
width = 0.18
for i, (col, label, color) in enumerate(wls):
    box_data = [df_ssa[df_ssa['Season'] == s][col].dropna() for s in SEASONS_ORDER]
    bp = ax.boxplot(box_data, positions=positions + i*width, widths=width*0.85,
                   patch_artist=True, showfliers=False)
    for patch in bp['boxes']:
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    # Invisible line for legend
    ax.plot([], [], 's', color=color, label=label, markersize=8)
ax.set_xticks(positions + 1.5*width)
ax.set_xticklabels(SEASONS_ORDER)
ax.set_ylabel('Single Scattering Albedo', fontsize=11)
ax.set_title('(a) SSA by Season and Wavelength', fontweight='bold')
ax.legend(fontsize=9, title='Wavelength')
ax.grid(True, alpha=0.3, axis='y')

# (b) dSSA (SSA_440 - SSA_870) by season
ax = axes[0, 1]
valid = df_ssa.dropna(subset=['dSSA'])
box_data = [valid[valid['Season'] == s]['dSSA'].dropna() for s in SEASONS_ORDER]
bp = ax.boxplot(box_data, labels=SEASONS_ORDER, patch_artist=True, showfliers=True,
                flierprops=dict(marker='.', markersize=3, alpha=0.3))
for patch, s in zip(bp['boxes'], SEASONS_ORDER):
    patch.set_facecolor(SEASON_COLORS[s])
    patch.set_alpha(0.7)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_ylabel('dSSA = SSA₄₄₀ − SSA₈₇₀', fontsize=11)
ax.set_title('(b) dSSA by Season', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# (c) SSA_440 time series
ax = axes[1, 0]
for season in SEASONS_ORDER:
    s = df_ssa[df_ssa['Season'] == season]
    ax.scatter(s.index, s['SSA_440'], s=15, alpha=0.5,
              color=SEASON_COLORS[season], label=season)
rolling = df_ssa['SSA_440'].rolling(30, min_periods=7).mean()
ax.plot(rolling.index, rolling.values, 'k-', linewidth=2, alpha=0.7, label='30-day mean')
ax.set_ylabel('SSA at 440nm', fontsize=11)
ax.set_title('(c) SSA₄₄₀ Time Series', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (d) SSA_440 vs BC
ax = axes[1, 1]
valid = df_ssa.dropna(subset=['IR BCc', 'SSA_440'])
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['IR BCc'], s['SSA_440'], s=30, alpha=0.5,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.2, label=season)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['IR BCc'], valid['SSA_440'])
    ax.text(0.05, 0.05, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
           transform=ax.transAxes, fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('BC (µg/m³)', fontsize=11)
ax.set_ylabel('SSA at 440nm', fontsize=11)
ax.set_title('(d) SSA₄₄₀ vs Surface BC', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('SSA Overview (AERONET Level 1.5 Inversion, Almucantar)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/ssa_overview.png', bbox_inches='tight')
plt.show()

print('\nSSA statistics by season:')
for s in SEASONS_ORDER:
    sub = df_ssa[df_ssa['Season'] == s]
    print(f'  {s}: n={len(sub)}, SSA440={sub["SSA_440"].mean():.3f}±{sub["SSA_440"].std():.3f}, '
          f'dSSA={sub["dSSA"].mean():.4f}±{sub["dSSA"].std():.4f}')

In [ ]:
# ── 8b. dSSA vs BC − Fine Soil (Eom et al. Figure) ──
# Merge filter chemistry with SSA data
chem_ssa = chem_pivot.set_index('date').copy()
chem_ssa.index = pd.to_datetime(chem_ssa.index)
chem_ssa = chem_ssa.join(ssa_df[['SSA_440', 'SSA_675', 'SSA_870', 'SSA_1020', 'dSSA']], how='inner')
chem_ssa['Month'] = chem_ssa.index.month
chem_ssa['Season'] = chem_ssa['Month'].map(SEASON_MAP)

bc_for_diff = 'ftir_ec' if 'ftir_ec' in chem_ssa.columns else 'hips_bc'
chem_ssa['BC_minus_FS'] = chem_ssa[bc_for_diff] - chem_ssa['fine_soil']

valid = chem_ssa.dropna(subset=['dSSA', 'BC_minus_FS'])
print(f'dSSA vs BC−FS: {len(valid)} matched filter-days with SSA')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# (a) dSSA vs BC - Fine Soil (the actual Eom et al. metric!)
ax = axes[0]
for season in SEASONS_ORDER:
    s = valid[valid['Season'] == season]
    ax.scatter(s['BC_minus_FS'], s['dSSA'], s=60, alpha=0.7,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
ax.axvline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
if len(valid) >= 5:
    r, p = stats.pearsonr(valid['BC_minus_FS'], valid['dSSA'])
    ax.text(0.05, 0.95, f'r={r:.3f}, p={p:.2e}\nn={len(valid)}',
           transform=ax.transAxes, va='top', fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel(f'{bc_for_diff} − Fine Soil (µg/m³)', fontsize=11)
ax.set_ylabel('dSSA = SSA₄₄₀ − SSA₈₇₀', fontsize=11)
ax.set_title('(a) dSSA vs BC−FS (Eom et al.)', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (b) SSA_440 vs BC/Fine Soil ratio
ax = axes[1]
valid2 = chem_ssa.dropna(subset=['SSA_440', bc_for_diff, 'fine_soil'])
valid2 = valid2[valid2['fine_soil'] > 0]
valid2['BC_FS_ratio'] = valid2[bc_for_diff] / valid2['fine_soil']
for season in SEASONS_ORDER:
    s = valid2[valid2['Season'] == season]
    ax.scatter(s['BC_FS_ratio'], s['SSA_440'], s=60, alpha=0.7,
              color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3, label=season)
ax.axvline(1, color='red', linewidth=1, linestyle='--', alpha=0.4, label='BC=FS')
if len(valid2) >= 5:
    r, p = stats.pearsonr(valid2['BC_FS_ratio'], valid2['SSA_440'])
    ax.text(0.95, 0.05, f'r={r:.3f}, p={p:.2e}',
           transform=ax.transAxes, ha='right', fontsize=9,
           bbox=dict(boxstyle='round', fc='white', alpha=0.9))
ax.set_xlabel('BC / Fine Soil', fontsize=11)
ax.set_ylabel('SSA at 440nm', fontsize=11)
ax.set_title('(b) SSA₄₄₀ vs BC/Dust Ratio', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# (c) SSA spectral shape by season: SSA_440, SSA_675, SSA_870, SSA_1020
ax = axes[2]
wavelengths = [440, 675, 870, 1020]
for season in SEASONS_ORDER:
    sub = chem_ssa[chem_ssa['Season'] == season]
    means = [sub[f'SSA_{w}'].mean() for w in wavelengths]
    stds = [sub[f'SSA_{w}'].std() for w in wavelengths]
    n = len(sub)
    if n > 0:
        ax.errorbar(wavelengths, means, yerr=[s/np.sqrt(n) for s in stds],
                    fmt='o-', markersize=8, linewidth=2, capsize=4,
                    color=SEASON_COLORS[season], label=f'{season} (n={n})')
ax.set_xlabel('Wavelength (nm)', fontsize=11)
ax.set_ylabel('SSA', fontsize=11)
ax.set_title('(c) SSA Spectral Shape by Season', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('dSSA and SSA Spectral Analysis (Eom et al. method, with AERONET L1.5)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUT_PLOTS}/dssa_bc_fine_soil_actual.png', bbox_inches='tight')
plt.show()

# Interpretation
print('\nInterpretation guide:')
print('  dSSA < 0: SSA decreases from 440→870nm → absorption increases at longer λ → dust-like')
print('  dSSA > 0: SSA decreases from 870→440nm → absorption increases at shorter λ → BC-dominated')
print(f'\nOverall dSSA: mean={valid["dSSA"].mean():.4f}, median={valid["dSSA"].median():.4f}')

In [ ]:
# ── 8c. w vs SSA at Multiple Wavelengths (Eom et al.) ──
# Does Addis follow the same w-SSA pattern as the other 14 SPARTAN sites?
# Higher w → more scattering-dominated → higher SSA

# Merge w data with SSA
w_ssa = df_w.set_index('date')[['w', bc_col_for_w, 'AS', 'AN', 'Season']].copy()
w_ssa = w_ssa.join(ssa_df[['SSA_440', 'SSA_675', 'SSA_870', 'SSA_1020']], how='inner')

print(f'w + SSA matched: {len(w_ssa)} days')

if len(w_ssa) >= 5:
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    for i, (wl, color) in enumerate([(440, '#9C27B0'), (675, '#4CAF50'),
                                       (870, '#F44336'), (1020, '#795548')]):
        ax = axes[i]
        col = f'SSA_{wl}'
        valid = w_ssa.dropna(subset=['w', col])
        
        for season in SEASONS_ORDER:
            s = valid[valid['Season'] == season]
            ax.scatter(s['w'], s[col], s=50, alpha=0.6,
                      color=SEASON_COLORS[season], edgecolors='k', linewidth=0.3,
                      label=season)
        
        if len(valid) >= 5:
            r, p = stats.pearsonr(valid['w'], valid[col])
            # Fit line
            z = np.polyfit(valid['w'], valid[col], 1)
            xr = np.linspace(valid['w'].min(), valid['w'].max(), 50)
            ax.plot(xr, np.poly1d(z)(xr), 'k--', linewidth=1.5, alpha=0.6)
            ax.text(0.05, 0.05, f'r={r:.3f}\np={p:.2e}\nn={len(valid)}',
                   transform=ax.transAxes, fontsize=9,
                   bbox=dict(boxstyle='round', fc='white', alpha=0.9))
        
        ax.set_xlabel('w = (AS+AN)/(AS+AN+BC)', fontsize=10)
        ax.set_ylabel(f'SSA at {wl}nm', fontsize=10)
        ax.set_title(f'{wl}nm', fontweight='bold', fontsize=12)
        if i == 0:
            ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle('w Parameter vs SSA at Multiple Wavelengths (Eom et al.)\n'
                 'Does Addis Ababa follow the global 14-site pattern?',
                 fontsize=13, fontweight='bold', y=1.04)
    plt.tight_layout()
    plt.savefig(f'{OUT_PLOTS}/w_vs_ssa_multiwavelength.png', bbox_inches='tight')
    plt.show()
    
    print('\nCorrelations (w vs SSA):')
    for wl in [440, 675, 870, 1020]:
        col = f'SSA_{wl}'
        v = w_ssa.dropna(subset=['w', col])
        if len(v) >= 5:
            r, p = stats.pearsonr(v['w'], v[col])
            print(f'  {wl}nm: r={r:.3f}, p={p:.2e}, n={len(v)}')
else:
    print('Not enough matched w + SSA data for analysis')

---
## Section 9: Future Work — Boundary Layer Height Stratification

**Method (Segura et al. 2017):** Mixing layer height dramatically affects PM-AOD
correlations. When stratified by BLH, correlations jumped from ~0.3 to >0.9
(for H > 1000m).

**Recommended data source: ERA5 reanalysis** from Copernicus CDS.
- Variable: `boundary_layer_height` (parameter 159)
- Resolution: 0.25 x 0.25 degrees, hourly
- Validated within ~130m of radiosondes (best among reanalyses)

**Setup required:**
1. Register at https://cds.climate.copernicus.eu/
2. Accept ERA5 terms of use
3. Create `~/.cdsapirc` with your API key
4. `pip install cdsapi`

In [ ]:
# ── ERA5 BLH Download Script (run once, then load from file) ──
# Uncomment to download ERA5 boundary layer height for Addis Ababa:

# import cdsapi
# client = cdsapi.Client()
#
# for year in ['2022', '2023', '2024']:
#     if year == '2022':
#         months = ['12']
#     elif year == '2023':
#         months = [f'{m:02d}' for m in range(1, 13)]
#     else:
#         months = [f'{m:02d}' for m in range(1, 8)]
#
#     client.retrieve(
#         'reanalysis-era5-single-levels',
#         {
#             'product_type': ['reanalysis'],
#             'variable': ['boundary_layer_height'],
#             'year': [year],
#             'month': months,
#             'day': [f'{d:02d}' for d in range(1, 32)],
#             'time': [f'{h:02d}:00' for h in range(24)],
#             'data_format': 'netcdf',
#             'area': [9.5, 38.2, 8.5, 39.2],  # N, W, S, E around Addis
#         },
#         f'{OUT_DATA}/era5_blh_addis_{year}.nc'
#     )
#     print(f'Downloaded ERA5 BLH for {year}')

# ── Once downloaded, load and process: ──
# import xarray as xr
#
# ds = xr.open_mfdataset(f'{OUT_DATA}/era5_blh_addis_*.nc')
# blh = ds['blh'].sel(latitude=9.0, longitude=38.7, method='nearest')
# blh_df = blh.to_dataframe().reset_index()
# blh_df['Date'] = blh_df['valid_time'].dt.normalize()
#
# # Daily stats: max BLH (afternoon peak), mean, min
# blh_daily = blh_df.groupby('Date')['blh'].agg(['max', 'mean', 'min']).rename(
#     columns={'max': 'BLH_max', 'mean': 'BLH_mean', 'min': 'BLH_min'})
#
# # Merge with master dataset
# master_blh = master.join(blh_daily, how='inner')
#
# # Bin by BLH (Segura et al. approach): <500m, 500-1000m, >1000m
# master_blh['BLH_bin'] = pd.cut(master_blh['BLH_max'],
#                                  bins=[0, 500, 1000, 5000],
#                                  labels=['<500m', '500-1000m', '>1000m'])
#
# # Redo BC vs AOD correlations within each BLH bin
# print('BC vs AOD correlations stratified by BLH (Segura et al.):')
# for blh_bin in ['<500m', '500-1000m', '>1000m']:
#     sub = master_blh[master_blh['BLH_bin'] == blh_bin]
#     valid = sub.dropna(subset=['IR BCc', 'AOD_500nm'])
#     if len(valid) >= 5:
#         r, p = stats.pearsonr(valid['IR BCc'], valid['AOD_500nm'])
#         print(f'  BLH {blh_bin}: r={r:.3f}, p={p:.2e}, n={len(valid)}')
#
# # Check if HIPS-FTIR discrepancy varies with BLH
# hf_blh = master_blh.dropna(subset=['hips_bc', 'ftir_ec'])
# hf_blh['hips_ftir_ratio'] = hf_blh['hips_bc'] / hf_blh['ftir_ec']
# print('\\nHIPS/FTIR ratio by BLH bin:')
# for blh_bin in ['<500m', '500-1000m', '>1000m']:
#     sub = hf_blh[hf_blh['BLH_bin'] == blh_bin]
#     if len(sub) >= 3:
#         print(f'  BLH {blh_bin}: ratio={sub["hips_ftir_ratio"].mean():.2f}'
#               f'±{sub["hips_ftir_ratio"].std():.2f}, n={len(sub)}')

print('ERA5 BLH stub ready — register at CDS and uncomment to download')

---
## Summary of Findings

| # | Analysis | Method | Key Question | Status |
|---|----------|--------|-------------|--------|
| 1 | BC/AOD sequential exclusion | Eom et al. | Does filtering low BC/AOD days improve correlations? | Complete |
| 2 | *w* parameter | Eom et al. | What fraction of fine mass is non-absorbing? | Complete |
| 3 | dAOD vs BC−FS | Eom (modified) | Is Addis BC-dominated or dust-dominated optically? | Complete |
| 4 | AOD binning | Segura et al. | Does binning reduce scatter in BC-AOD? | Complete |
| 5 | FMF seasonal | Eom et al. | Do coarse fractions shift by season? | Complete |
| 6 | η decomposition | Snider et al. | What drives PM₂.₅/AOD variability? | Complete (BC proxy) |
| 7 | HIPS-FTIR stratification | — | Does discrepancy vary with AOD/BC-AOD? | Complete |
| 8 | SSA analysis | Eom et al. | dSSA vs BC-FS, w vs SSA, SSA spectral shape | Complete (L1.5) |
| 9 | BLH stratification | Segura et al. | Does BLH explain BC-AOD decoupling? | Stub (need ERA5) |

### Data Notes
- **SSA data:** Level 2.0 is empty (AOD₄₄₀ threshold of 0.4 not met). Using **Level 1.5**
  (cloud-cleared, quality-controlled, ~629 daily obs). Caveat: not fully quality-assured.
- **BLH data:** ERA5 reanalysis recommended (0.25° hourly). Requires free CDS registration.

### Next Step
Register at https://cds.climate.copernicus.eu/, download ERA5 BLH, and run Section 9.

In [ ]:
# Save master dataset for follow-up analysis
master.to_csv(f'{OUT_DATA}/master_surface_columnar.csv')
print(f'Master dataset saved: {len(master)} rows, {len(master.columns)} columns')
print(f'\nPlots saved to: {OUT_PLOTS}/')
print('\n' + '='*80)
print('SURFACE–COLUMNAR LINKAGE ANALYSIS COMPLETE')
print('='*80)